In [1]:
from pathlib import Path
import glob
import json
import os
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat


In [2]:

# Ingestion

pdf_paths = glob.glob("../data/raw/pdf/*.pdf")

pdf_titles = []

for pdf in pdf_paths:
    fp = Path(pdf)
    filename = fp.stem
    pdf_titles.append(filename)


pdf_ids = list(range(len(pdf_titles)))

display(pdf_titles)

['Administrative Resolution No. (1) of 2025 for Commercial Maritime',
 'Administrative Resolution No. (351) of 2026 Concerning the Booking of Appointments foInspection Service',
 'Administrative Resolution No. (363) of 2026 Concerning the Rules and Criteria for Designating Peak Hours',
 'Administrative Resolution No. (9) of 2025 disclosure Maritime Container Services',
 'Administrative Resolution No. (939) of 2025 Issuing the Operation of Autonomous Vehicles',
 'Executive Council Resolution No. (1) of 2026 Technical Inspection',
 'Executive Council Resolution No. (14) of 2026 Approving the Use of Autonomous Vehicles',
 'Executive Council Resolution No. (15) of 202 Applicable to Operators Civil Aviation-related Professions and Activities',
 'Executive Council Resolution No. (2) of 2026 the Activity of Driving Instruction',
 'Executive Council Resolution No. (3) of 2026 Concerning for Public Transport']

In [3]:
# Ingestion pipeline
from docling_core.types import DoclingDocument

docs = []
docs_dict = []
processed_dir = Path("../data/processed")

for pdf in pdf_paths:
    processed_path = processed_dir / f"{Path(pdf).stem}.json"
    if not processed_path.exists():
        print(f"Processed file not found: {processed_path.resolve()}")
        continue

    with processed_path.open("r", encoding="utf-8") as f:
        doc_dict = json.load(f)

    docs_dict.append(doc_dict)
    docs.append(DoclingDocument.model_validate(doc_dict))


In [4]:
print(f"Ids length: {len(pdf_ids)}")
print(f"dict length: {len(docs_dict)}")

Ids length: 10
dict length: 10


In [11]:
# Function to serialize table to text properly
import pandas as pd

# for one table
def table_to_md(table):
    table_df : pd.DataFrame = table.export_to_dataframe(doc=doc) # table_df is the variable name, :pd is the type hint
    table_md = table_df.to_markdown()



In [33]:
# Chunking Function
import re
from docling.chunking import HybridChunker
from langchain_core.documents import Document
import pandas as pd

langchain_docs = []
chunker = HybridChunker(max_tokens = 512, repeat_table_header=True)

for i in range(len(docs)):
        
    # Document-level info
    document_id = pdf_ids[i]
    title = pdf_titles[i]

    # Type and year extraction
    type = title.rsplit('No.')[0].strip()
    print(type)
    
    if (re.search(r"\b(19|20)\d{2}\b", title)):
        year = int(re.search(r"\b(19|20)\d{2}\b", title).group())
    else:
        "Unknown"
    
    print(year)

    chunk_iter = chunker.chunk(dl_doc = docs[i]) # list of chunks per doc
        
        
    for j, chunk in enumerate(chunk_iter):
        # Chunk-level info
        chunk_id = f"{title}_{j}"
        text = chunker.contextualize(chunk)

        page = None
        if chunk.meta.doc_items:
            prov = chunk.meta.doc_items[0].prov
            if prov:
                page = prov[0].page_no

        headings = chunk.meta.headings
        captions = chunk.meta.captions            

        chunk_metadata = {
            'document_id': document_id,
            'year': year,
            'type': type,
            'chunk_id': chunk_id,
            'filename': title,
            'page': page,
            'headings': headings,
            'captions': captions,
        }

        langchain_docs.append(
            Document(
                page_content=text,
                metadata = chunk_metadata))




Administrative Resolution
2025
Administrative Resolution
2026
Administrative Resolution
2026
Administrative Resolution
2025


C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (682 > 512). Running this sequence through the model will result in indexing errors
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions


Administrative Resolution
2025
Executive Council Resolution
2026


C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions


Executive Council Resolution
2026


C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions


Executive Council Resolution
2026
Executive Council Resolution
2026
Executive Council Resolution
2026


C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions
C:\Users\yarah\AppData\Local\Temp\ipykernel_1912\1032868005.py:42: DeprecationWarning: deprecated
  captions = chunk.meta.captions


In [34]:
# # To check that there are no tokens that were larger than 512
# oversized = [
#     (d.metadata.get("chunk_id"), chunker.tokenizer.count_tokens(d.page_content))
#     for d in langchain_docs
#     if chunker.tokenizer.count_tokens(d.page_content) > chunker.max_tokens
# ]
# print(oversized)

[]


In [38]:
# # Checking the chunks

# for doc in langchain_docs[:20]:
#     print(doc.metadata["chunk_id"])


In [ ]:
# Google API Key, this will be used for embedding + LLM
import getpass
if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_LEY"] = getpass.getpass("Enter your google API key:")

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
# gemini-embedding-2-preview
text = [document.page_content for document in langchain_docs]

# Only embed the enriched text, the metadata is attached to the doc


In [40]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="road_and_transport_collection",
    embedding_function=embeddings,
    persist_directory="../chroma_langchain_db"
)

In [41]:
import time

batch_size = 10
delay = 10
num = len(langchain_docs)

for start in range(0, num, batch_size):
    batch = langchain_docs[start:start+batch_size]
    batch_ids = [doc.metadata["chunk_id"] for doc in batch]

    vector_store.add_documents(
        documents= batch,
        ids= batch_ids
    )
    i = i + batch_size
    time.sleep(delay)

In [94]:
question = "How long does the DMA take to approve a permit?"


In [71]:
# BM25 Retriever

from langchain_community.retrievers import BM25Retriever

retriever = BM25Retriever.from_documents(
    documents= langchain_docs,
    k= 5
)


In [72]:
results_semantic = vector_store.similarity_search_by_vector(
    embedding = embeddings.embed_query(user_query),
    k=5
)
results_BM25 = retriever.invoke(user_query)

In [ ]:
def RRF_score(semantic_res, BM25_res, rrf_k=60):
    scores = {}

    for results in (semantic_res, BM25_res):
        for rank, doc in enumerate(results, start=1):
            chunk_id = doc.metadata["chunk_id"]
            scores[chunk_id] = scores.get(chunk_id, 0) + 1 / (rrf_k + rank)

    sorted_scores= sorted(
         scores.items(),
         key = lambda item: item[1],
         reverse= True # largest to smallest
    )

    return sorted_scores

In [89]:
reranked_results = RRF_score(semantic_res=results_semantic, BM25_res=results_BM25)

# Top k results
top_k = 5
top_results = reranked_results[:top_k]

doc_lookup = {
    doc.metadata["chunk_id"]: doc for doc in langchain_docs
}

top_chunks = [
    doc_lookup[chunk_id] for chunk_id, score in top_results if chunk_id in doc_lookup
]


In [90]:
print(top_chunks)

[Document(metadata={'document_id': 0, 'year': 2025, 'type': 'Administrative Resolution', 'chunk_id': 'Administrative Resolution No. (1) of 2025 for Commercial Maritime_8', 'filename': 'Administrative Resolution No. (1) of 2025 for Commercial Maritime', 'page': 5, 'headings': ['Exemption from Certain Requirements Article (6)'], 'captions': None}, page_content='Exemption from Certain Requirements Article (6)\nA Vessel may, in special and justifiable cases as dictated by the public interest, be exempted from certain Permit issuance conditions or requirements, provided that this exemption does not compromise Maritime Safety or the marine environment in the Waters of the Emirate, and is subject to a decision issued by the Executive Director of the DMA.'), Document(metadata={'document_id': 0, 'year': 2025, 'type': 'Administrative Resolution', 'chunk_id': 'Administrative Resolution No. (1) of 2025 for Commercial Maritime_7', 'filename': 'Administrative Resolution No. (1) of 2025 for Commercia

In [99]:
# sysem messages
system_message = """You are an assistant for answering questions about Roads and Transport
        legislation and related documents.
        
        You must answer the user's question using only the evidence provided to you.
        
        Rules:
        1. Base your answer only on provided retrieved context.
        2. Do not use outside knowledge or make assumptions.
        3. Preserve important legal, regulatory, and technical details from the source.
        4. If the provided evidence is insufficient, clearly state so.
        5. Do not invent legislation, requirements, dates, authorities, document details, or citations.
        6. When evidence from multiple chunks is relevant, combine it into a clear and coherent answer.
        7. If retrieved sources conflict, explicitly mention the conflict rather than choosing one.
        8. Keep the answer focused on the user's question.
        """

human_message = """Retrieved evidence:
     {context}
     
     Question:
     {question}
     
     Answer:"""

In [101]:
context = "\n\n".join(chunk.page_content for chunk in top_chunks)

from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", system_message),
    ("human", human_message)
])

In [97]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model = "gemini-2.5-flash",
    temperature=0.1
)

In [104]:

formatted_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(formatted_prompt)

In [105]:
# my answer node

print(response)

content='The provided evidence does not state how long the DMA takes to approve a permit.\n\nHowever, it does specify that:\n*   The RTA will determine a Permit application within ten (10) working days from the date of its submission, provided it meets all relevant conditions and requirements.\n*   The DMA will issue a decision approving Fees within thirty (30) days from the date on which the application is submitted, unless there are valid justifications for extending this time frame.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1e088-beb5-74df-b915-99bc035f192c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1169, 'output_tokens': 928, 'total_tokens': 2097, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 830}}


In [106]:
chain = prompt | llm

c_response = chain.invoke({
    "context": context,
    "question": question
})

print(c_response)

content='The provided evidence states that the RTA will determine a permit application within ten (10) working days from the date of its submission. However, the evidence does not specify how long the DMA takes to approve a permit. It does state that the DMA will issue a decision approving fees within thirty (30) days from the date on which the application for fee approval is submitted.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a1e08b-35b2-714a-a53e-34813ea1bfd9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 1169, 'output_tokens': 810, 'total_tokens': 1979, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 733}}
